# Recurrent Neural Networks

In this chapter, we will deal with variable-length sequence data. 
This is fundamentally different from fixed shape data that we have previously encountered. But variable-length data is abundant in the real-world. Tasks such as translating passages of text from one language to another, 
engaging in dialogue, or controlling a robot, demand that models both ingest and output sequentially structured data. Here we focus on text data which is our primary interest. In particular, we sample character sequences as training data from a dataset of Spanish names. To increase complexity, we continue with *The Time Machine* (1895) by [H. G. Wells](https://en.wikipedia.org/wiki/H._G._Wells).

For sequence modeling, we explore two approaches. In the previous chapter, we used fixed-length windows, or **contexts**, to predict the probability distribution of the next token. This allows us to use familiar models such as CNNs and MLPs. Here we consider processing sequences of arbitrary length. In particular, we introduce **Recurrent Neural Networks** (RNNs) which are neural networks that capture the dynamics of sequences via *recurrent connections* (@fig-04-rnn), which can be thought of as cycles that iteratively updates a latent state vector. 
The resulting hidden representation depend on the specific input order. Hence, RNNs inherit causality from the structure of the text.

To understand the challenges of training RNNs, we derive the **BPTT equations** (**B**ack**p**ropagation **T**hrough **T**ime). We will see that RNNs accumulate gradients with depth corresponding to time steps, instead of number of layers for MLPs[^path-length]. In particular, we will see that RNNs struggle to model long-term dependencies (i.e., tokens that are spaced far apart but share a significant relationship) which manifest as vanishing gradient. This had motivated the development of more advanced RNN architectures (e.g., LSTM [@lstm] and GRU [@gru]) that aim to minimize or address vanishing gradients.

[^path-length]: Both can be formulated in terms of the path length in the computation graph between two nodes that share a dependency.

# RNN  cell

Previously, we described various language models where the conditional probability of token $\mathbf{x}_t$ depends on a fixed context $\mathbf{x}_{[t - \tau: t-1]}.$ If we want to incorporate the possible effect of tokens earlier than the given context, we need to increase the context size $\tau$. For the *n*-gram model, this would increase the parameters exponentially in $\tau$. Using embeddings, the MLP network the number of parameters grows as $O(\tau)$. Finally, using convolutions this decreases to $O(\log \tau).$
Alternatively, instead of modeling the next token directly in terms of previous tokens, we can use a latent variable that, in principle, stores *all* previous information up to the previous time step:

$$
p(\mathbf{x}_{t} \mid \mathbf{x}_{1}, \ldots, \mathbf{x}_{t-1}) \approx p(\mathbf{x}_{t} \mid \mathbf{h}_{t-1})
$$

where $\mathbf{h}_{t-1}$ is a *hidden state* that stores information up to the time step $t - 1.$ The hidden state is updated based on the current input and the previous state: 

$$
\mathbf{h}_{t} = f(\mathbf{x}_{t}, \mathbf{h}_{t-1})
$$

so that $\mathbf{h}_{t} = F(\mathbf{x}_{1}, \ldots, \mathbf{x}_{t}, \mathbf{h}_{0})$ for some $\mathbf{h}_{0}$ where $F$ involves recursively applying $f$ (see @fig-04-rnn). For a sufficiently complex function $f$, the above latent variable model is not an approximation, since $\mathbf{h}_{t}$ can simply store all $\mathbf{x}_{1}, \ldots, \mathbf{x}_{t}$ it has observed so far. In our case, we use fully-connected layers whose complexity can be tuned with its width.

<br>

![RNN unit (a) cyclic, and (b) unrolled RNN (essentially a deep MLP with shared weights).](img/04-rnn.svg){#fig-04-rnn fig-align="center" width=600px}

**RNN cell.** Let each token be represented by vectors $\mathbf{x}_t \in \mathbb{R}^{d}$ and let $\mathbf{h}_0 = \boldsymbol{0}.$ Then,

$$
\begin{aligned}
\mathbf{h}_t &= \tanh(\mathbf{x}_t \mathbf{U} + \mathbf{h}_{t-1} \mathbf{W} + \mathbf{b}) \\
\mathbf{y}_t &= \mathbf{h}_t
\end{aligned}
$$

where $\mathbf{U} \in \mathbb{R}^{d \times h}$, $\mathbf{W} \in \mathbb{R}^{h \times h}$, and $\mathbf{b} \in \mathbb{R}^{h}.$ Here $h$ is the dimensionality of the hidden state. For character-level inputs, $\mathbf{x}_t$ can be a one-hot vector of length $|\mathcal{V}|$ so that $\mathbf{U}$ is $|\mathcal{V}| \times h$, also acting as the embedding matrix for the tokens[^large-vocab]. Finally, the state vector is also the **output** at each step, used by downstream layers of the network. The computation is illustrated in @fig-04-simple-rnn.

[^large-vocab]: RNNs with input matrix $\mathbf{U}$ of size $|\mathcal{V}| \times h$ does not work with large vocabularies. Vocabulary size $|\mathcal{V}|$ can be very large (e.g., tens of thousands or more). If you directly use one-hot encoded vectors of size $|\mathcal{V}|$, the cell input would be extremely high-dimensional and sparse. Instead of performing a full matrix multiplication, the embedding layer simply indexes into the embedding matrix to retrieve the corresponding dense vector for each word of size $d_\text{emb}.$ Then, $\mathbf{U}$ has shape $d_\text{emb} \times h.$

<br>

![Computational graph of an unrolled simple RNN. [Source](https://www.d2l.ai/chapter_recurrent-neural-networks/rnn.html)](img/04-simple-rnn.svg){#fig-04-simple-rnn fig-align="center" width=600px}

**Remark.** RNNs use the same parameters at each time step, i.e. it is assumed that the dynamics is *stationary*. Practically, this means that the parameter count does not grow as the sequence length increases, and that the parameters have to time index.

## Code implementation

First, we implement the recurrent layer. To implement batch computation, an input $\mathbf{X}$ has shape $(T, B, d).$ That is, a batch of $B$ sequences of length $T$, consisting of vectors in $\mathbb{R}^{d}.$ Elements of a batch are computed independently, ideally in parallel. For example, $\mathbf{X}_{[0, :, :]}$ consist of a batch of all vectors at $t = 0.$ Similarly, $\mathbf{X}_{[:, 0, :]}$ is one instance of a sequence of vectors. At each step, the layer returns the state vector of shape $(B, h).$ These are stacked to get a tensor of shape $(T, B,h)$ consistent with the input. RNN cell computation can be written as:

```python
outs = []
for t in range(T):
    h = torch.tanh(x[t] @ self.U + h @ self.W + self.b)
    outs.append(h)
```

Here the state `h`is updated at each step, and the output vector is also set to `h` at each step. The initialization of the state vector is not shown, but we typically set it to zero when not specified.
This leads us to the required methods in the base class below. But first, let us define the expected "shape" of an RNN unit.

### Base RNN

A *recurrent unit* is any function that iteratively updates a state based on new sequence input, it may haver other layers for downstream processing at each time step, so we also return an output tensor. We set the following guidelines:

1. A recurrent unit must have `(inputs_dim, hidden_dim, **kwargs)` as arguments.
2. It's forward signature is `(x, state=None)` where `x` is "sequence first", i.e. $(T, B, d)$.
3. It's forward return format is `outs, state` where `state` has the expected format as input for the forward function and `outs` has shape $(T, B, h)$.

The parameter `(d, h)` for an RNN can be read off as transforming the each sequence element from $\mathbb{R}^d$ to $\mathbb{R}^h.$ Next, an implementation with expected inputs $(T, B, d)$ is already linear layer friendly. This can be interpreted as processing an entire batch at each time step, instead of entire sequences per batch. Hence, we choose this over the more intuitive "batch first" shape $(B, T, d)$. 

Finally, while the output has to be of shape $(T, B, h)$, the `state` is more arbitrary. For example, it can be a tuple of tensors `(h, c)`. As such, we can call the unit with either `(x)` or `(x, state=(h0, c0))`. This latter is useful for setting up a warmup state, or continuing inference with another input sequence. The only constraint is that the states are consistently formatted in all parts of a specific implementation. For example, if `state` is the current output state, then the unit can be called next with `(x, state=state)` without errors. 

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import random

RANDOM_SEED = 0
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
MPS = torch.backends.mps.is_available()
CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda:0") if CUDA else torch.device("mps") if MPS else torch.device("cpu")

In [ ]:
class RNNBase(nn.Module):
    """Base class for recurrent units, e.g. RNN, LSTM, GRU, etc."""
    def __init__(self, inputs_dim: int, hidden_dim: int):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.inputs_dim = inputs_dim
        
    def init_state(self, x):
        raise NotImplementedError
    
    def compute(self, x, state):
        raise NotImplementedError

    def forward(self, x, state=None):
        state = self.init_state(x) if state is None else state
        outs, state = self.compute(x, state)
        return outs, state

Implementing the basic RNN cell:

In [ ]:
class RNN(RNNBase):
    """Simple RNN unit."""
    def __init__(self, inputs_dim: int, hidden_dim: int):
        super().__init__(inputs_dim, hidden_dim)
        self.U = nn.Parameter(torch.randn(inputs_dim, hidden_dim) / np.sqrt(inputs_dim))
        self.W = nn.Parameter(torch.randn(hidden_dim, hidden_dim) / np.sqrt(hidden_dim))
        self.b = nn.Parameter(torch.zeros(hidden_dim))

    def init_state(self, x):
        B = x.shape[1]
        h = torch.zeros(B, self.hidden_dim, device=x.device)
        return h
    
    def compute(self, x, state):
        h = state
        T = x.shape[0]
        outs = []
        for t in range(T):
            h = torch.tanh(x[t] @ self.U + h @ self.W + self.b)
            outs.append(h)
        return torch.stack(outs), h

**Remark.** It's important to note that our RNN does not store state outside of forward pass.

Shapes test, i.e. $(T, B, d)$ to $(T, B, h)$ for a network with params $(d, h)$:

In [ ]:
B, T, d, h = 32, 10, 30, 5
x = torch.randn(T, B, d)
rnn = RNN(d, h)
outs, state = rnn(x)
assert outs.shape == (T, B, h)
assert state.shape == (B, h)
assert torch.abs(outs[-1] - state).max() < 1e-8

<br>

**Remark.** The PyTorch RNN module has a similar API:

In [ ]:
B, T, d, h = 32, 10, 30, 5
rnn_torch = nn.RNN(d, h)
outs, state = rnn_torch(x)
assert outs.shape == (T, B, h)
assert state.shape == (1, B, h)
assert torch.abs(outs[-1] - state).max() < 1e-8

Correctness:

In [ ]:
for name, p in rnn.named_parameters():
    if name == "b":
        p.data.fill_(0.0)
    else:
        p.data.fill_(1.0)

for name, p in rnn_torch.named_parameters():
    if "bias" in name:
        p.data.fill_(0.0)
    else:
        p.data.fill_(1.0)

error = torch.abs(rnn(x)[0] - rnn_torch(x)[0]).max()
print(error)
assert error < 1e-6

# RNN language model

Our goal in this section is to train a character-level RNN language model to predict the next token at *each* step with varying-length context. Hence, during training, our model predicts on each time-step (@fig-04-char-rnn). The language model below is simply an RNN cell with an attached **logits layer** applied at each step.


![Character-level RNN language model for predicting the next character at each step.  [Source](https://www.d2l.ai/chapter_recurrent-neural-networks/rnn.html)](img/04-char-rnn.svg){#fig-04-char-rnn fig-align="center" width=550px}

To implement a language model, we simply attach a linear layer on the RNN unit to compute logits. 
The linear layer performs matrix multiplication on the rightmost dimension of `outs` which contains the value of the state vector at each time step. Thus, as shown in @fig-04-char-rnn we have $T$ predictions with increasing context size[^teacher-forcing] $1, 2, \ldots, T.$

[^teacher-forcing]: Consequently, the model gets corrected at each time step, with variable-length dependency, during backward pass. 

In [ ]:
import torch
import torch.nn as nn
from typing import Type
from functools import partial


class RNNLanguageModel(nn.Module):
    def __init__(self, 
        cell: Type[RNNBase],
        inputs_dim: int,
        hidden_dim: int,
        vocab_size: int,
        **kwargs
    ):
        super().__init__()
        self.cell = cell(inputs_dim, hidden_dim, **kwargs)
        self.linear = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, state=None, return_state=False):
        outs, state = self.cell(x, state)
        outs = self.linear(outs)    # (T, B, H) -> (T, B, C)
        return outs if not return_state else (outs, state)


LanguageModel = lambda cell: partial(RNNLanguageModel, cell)

<br>

## Character sequences dataset

Our dataset consists of $T$ input-output pairs of characters **shifted** one time step:

In [ ]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class SequenceDataset(Dataset):
    def __init__(self, data: torch.Tensor, seq_len: int, vocab_size: int):
        super().__init__()
        self.data = data
        self.seq_len = seq_len
        self.vocab_size = vocab_size

    def __getitem__(self, i):
        c = self.data[i: i + self.seq_len + 1]
        x, y = c[:-1], c[1:]
        x = F.one_hot(x, num_classes=self.vocab_size).float()
        return x, y
    
    def __len__(self):
        return len(self.data) - self.seq_len

Training on the *Time Machine* text:

In [ ]:
import re
import os
import torch
import requests
from collections import Counter
from typing import Union, Optional, TypeVar, List

from pathlib import Path

DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)


T = TypeVar("T")
ScalarOrList = Union[T, List[T]]


class Vocab:
    def __init__(self, 
        text: str, 
        min_freq: int = 0, 
        reserved_tokens: Optional[List[str]] = None,
        preprocess: bool = True
    ):
        text = self.preprocess(text) if preprocess else text
        tokens = list(text)
        counter = Counter(tokens)
        reserved_tokens = reserved_tokens or []
        self.token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True)
        self.itos = [self.unk_token] + reserved_tokens + [tok for tok, f in filter(lambda tokf: tokf[1] >= min_freq, self.token_freqs)]
        self.stoi = {tok: idx for idx, tok in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)
    
    def __getitem__(self, tokens: ScalarOrList[str]) -> ScalarOrList[int]:
        if isinstance(tokens, str):
            return self.stoi.get(tokens, self.unk)
        else:
            return [self.__getitem__(tok) for tok in tokens]

    def to_tokens(self, indices: ScalarOrList[int]) -> ScalarOrList[str]:
        if isinstance(indices, int):
            return self.itos[indices]
        else:
            return [self.itos[int(index)] for index in indices]
            
    def preprocess(self, text: str):
        return re.sub("[^A-Za-z]+", " ", text).lower().strip()

    @property
    def unk_token(self) -> str:
        return "▮"

    @property
    def unk(self) -> int:
        return self.stoi[self.unk_token]

    @property
    def tokens(self) -> List[int]:
        return self.itos


class Tokenizer:
    def __init__(self, vocab: Vocab):
        self.vocab = vocab

    def tokenize(self, text: str) -> List[str]:
        UNK = self.vocab.unk_token
        tokens = self.vocab.stoi.keys()
        return [c if c in tokens else UNK for c in list(text)]

    def encode(self, text: str) -> torch.Tensor:
        x = self.vocab[self.tokenize(text)]
        return torch.tensor(x, dtype=torch.int64)

    def decode(self, indices: Union[ScalarOrList[int], torch.Tensor]) -> str:
        return "".join(self.vocab.to_tokens(indices))

    @property
    def vocab_size(self) -> int:
        return len(self.vocab)


class TimeMachine:
    def __init__(self, download=False, path=None):
        DEFAULT_PATH = str((DATA_DIR / "time_machine.txt").absolute())
        self.filepath = path or DEFAULT_PATH
        if download or not os.path.exists(self.filepath):
            self._download()
        
    def _download(self):
        url = "https://www.gutenberg.org/cache/epub/35/pg35.txt"
        print(f"Downloading text from {url} ...", end=" ")
        response = requests.get(url, stream=True)
        response.raise_for_status()
        print("OK!")
        with open(self.filepath, "wb") as output:
            output.write(response.content)
        
    def _load_text(self):
        with open(self.filepath, "r") as f:
            text = f.read()
        s = "*** START OF THE PROJECT GUTENBERG EBOOK THE TIME MACHINE ***"
        e = "*** END OF THE PROJECT GUTENBERG EBOOK THE TIME MACHINE ***"
        return text[text.find(s) + len(s): text.find(e)]
    
    def build(self, vocab: Optional[Vocab] = None):
        self.text = self._load_text()
        vocab = vocab or Vocab(self.text)
        tokenizer = Tokenizer(vocab)
        encoded_text = tokenizer.encode(vocab.preprocess(self.text))
        return encoded_text, tokenizer

In [ ]:
from torch.utils.data import random_split

def collate_fn(batch):
    """Transforming the data to sequence-first format."""
    x, y = zip(*batch)
    x = torch.stack(x, 1)      # (T, B, vocab_size)
    y = torch.stack(y, 1)      # (T, B)
    return x, y


data, tokenizer = TimeMachine().build()
VOCAB_SIZE = tokenizer.vocab_size
dataset = SequenceDataset(data, seq_len=10, vocab_size=VOCAB_SIZE)
train_dataset, valid_dataset = random_split(dataset, [0.80, 0.20])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

The batch index (i.e. starting point) is shuffled, but the ordering in each sequence is intact:

In [ ]:
x, y = next(iter(train_loader))

a, T = 1, dataset.seq_len
x_chars = tokenizer.decode(torch.argmax(x[:, a], dim=1))     # inputs are one-hot
y_chars = tokenizer.decode(y[:, a])
for i in range(T):
    print(f"{x_chars[i]} --> {y_chars[i]}")

In [ ]:
print(x.shape, y.shape)
print("inputs:", torch.argmax(x[:, 0], dim=-1))
print("target:", y[:, 0])

PyTorch `F.cross_entropy` expects input `(B, C, T)` and target `(B, T)`:

In [ ]:
import torch.nn.functional as F

x, y = next(iter(train_loader))
model = LanguageModel(RNN)(VOCAB_SIZE, 5, VOCAB_SIZE)
loss = F.cross_entropy(model(x).permute(1, 2, 0), y.transpose(0, 1))
loss

# RNN model training

Each mini-batch contains $B \times T$ prediction instances for training. 
Prediction is done at each step, with the state updated at each step as well.
Below are logits for $t = 1, \ldots, 30.$ At each step, the hidden state is updated before making the next prediction. Finally, the model is evaluated at every time step with varying-length inputs. State starts at zero, so it may be necessary to warm the model up.

In [ ]:
def collate_fn(batch):
    """Transforming the data to sequence-first format."""
    x, y = zip(*batch)
    x = torch.stack(x, 1)      # (T, B, vocab_size)
    y = torch.stack(y, 1)      # (T, B)
    return x, y

Training on sequences from *The Time Machine*:

In [ ]:
from torch.utils.data import random_split

data, tokenizer = TimeMachine().build()
T = 30
BATCH_SIZE = 128
VOCAB_SIZE = tokenizer.vocab_size

dataset = SequenceDataset(data, seq_len=T, vocab_size=VOCAB_SIZE)
train_dataset, valid_dataset = random_split(dataset, [0.80, 0.20])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)    # also sampled

print("preds per epoch")
print("train:", f"{len(train_loader) * BATCH_SIZE * T: .3e}")
print("valid:", f"{len(valid_loader) * BATCH_SIZE * T: .3e}")

When training RNNs, it is common to use **gradient clipping**. RNNs are deep in another sense, i.e. in sequence length since we apply the state update function $f$ for each sequence element. Hence, during BP, we get matrix products of length $O(T).$ This causes gradients to explode or vanish resulting in numerical instability. A direct solution to exploding gradients is simply to clip them. Here we project them to a ball of radius $\xi > 0.$ Thus,

$$
\mathbf{g} \leftarrow \min \left(1, \frac{\xi}{\| \mathbf{g} \|} \right) \mathbf{g} = \min \left({\| \mathbf{g}\|,\, {\xi}} \right) \frac{\mathbf{g}}{\| {\mathbf{g}} \|}.
$$

First, the gradient is still in the same direction but clipped in norm to $\xi.$ 
So when $\| \mathbf{g} \| \leq \xi$, the gradient is unchanged. On the other hand, when 
$\| \mathbf{g} \| > \xi$, the above ratio goes out of the $\min$ operation, and the gradient is scaled to have norm $\xi.$

In [ ]:
import torch.nn.functional as F

def clip_grad_norm(model, max_norm: float):
    """Calculate norm on concatenated params. Modify params in-place."""
    params = [p for p in model.parameters() if p.requires_grad]
    norm = torch.sqrt(sum(torch.sum((p.grad ** 2)) for p in params))
    if norm > max_norm:
        for p in params:
            p.grad[:] *= max_norm / norm   # [:] = shallow copy, in-place

PyTorch `F.cross_entropy` expects input `(B, C, T)` and target `(B, T)`:

In [ ]:
def train_step(model, optim, x, y, max_norm) -> float:
    target = y.transpose(0, 1)
    output = model(x).permute(1, 2, 0)
    loss = F.cross_entropy(output, target)
    loss.backward()
    
    clip_grad_norm(model, max_norm=max_norm)
    optim.step()
    optim.zero_grad()
    return loss.item()


@torch.no_grad()
def valid_step(model, x, y) -> float:
    target = y.transpose(0, 1)
    output = model(x).permute(1, 2, 0)
    loss = F.cross_entropy(output, target)
    return loss.item()

Training the model:

In [ ]:
from tqdm.notebook import tqdm

DEVICE = "cpu"
LR = 0.01
EPOCHS = 5
MAX_NORM = 1.0

model = LanguageModel(RNN)(VOCAB_SIZE, 64, VOCAB_SIZE)
model.to(DEVICE)
optim = torch.optim.Adam(model.parameters(), lr=LR)

train_losses = []
valid_losses = []
for e in tqdm(range(EPOCHS)):
    for t, (x, y) in enumerate(train_loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        loss = train_step(model, optim, x, y, MAX_NORM)
        train_losses.append(loss)

        if t % 5 == 0:
            xv, yv = next(iter(valid_loader))
            xv, yv = xv.to(DEVICE), yv.to(DEVICE)
            valid_losses.append(valid_step(model, xv, yv))

print(np.array(valid_losses)[-5:].mean())

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats("svg")

plt.figure(figsize=(10, 4))
plt.plot(train_losses, label="train")
plt.plot(np.array(range(1, len(valid_losses) + 1)) * 5, valid_losses, label="valid")
plt.grid(linestyle="dotted", alpha=0.6)
plt.ylabel("loss")
plt.xlabel("step")
plt.legend();

Note we addressed problem of exploding gradients, but not vanishing gradients.

In [ ]:
!mkdir -p artifacts/
PATH = "./artifacts/rnn_lm.pkl"
torch.save(model.state_dict(), PATH)

# Text generation

Recall that the state vector is initialized as zero. So we use a **warmup context** or a **prompt** to allow the RNN cell to update its state iteratively by processing one character at a time from the warmup text. Then, the algorithm simulates the prediction process of our RNN language model, but instead of using a predefined input sequence, it uses the *previous output* as the next input.

<br>

![An input sequence is used to get a final state vector (this is the warmup stage, i.e. the state goes from zero to some nonzero vector). The final character and state during warmup is used to predict the next character. This process is repeated until the number of predicted tokens is reached.](img/04-rnn-textgen.png){#fig-04-rnn-textgen fig-align="center" width=500px}

Loading the trained RNN language model:

In [ ]:
DEVICE = "cpu"  # faster for RNN inference
WEIGHTS_PATH = "./artifacts/rnn_lm.pkl"
data, tokenizer = TimeMachine().build()
VOCAB_SIZE = tokenizer.vocab_size

model = LanguageModel(RNN)(VOCAB_SIZE, 64, VOCAB_SIZE)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE));

Text generation utils and algorithm:

In [ ]:
import torch
import torch.nn.functional as F

class TextGenerator:
    def __init__(self, model, tokenizer, device="cpu"):
        self.model = model.to(device)
        self.device = device
        self.tokenizer = tokenizer

    def _inp(self, indices: list[int]):
        """Preprocess indices (T,) to (T, 1, V) shape with B=1."""
        VOCAB_SIZE = self.tokenizer.vocab_size
        x = F.one_hot(torch.tensor(indices), VOCAB_SIZE).float()
        return x.view(-1, 1, VOCAB_SIZE).to(self.device)

    @staticmethod
    def sample_token(logits, temperature: float):
        """Convert logits to probs with softmax temperature."""
        p = F.softmax(logits / temperature, dim=1)  # T = ∞ => exp ~ 1 => p ~ U[0, 1]
        return torch.multinomial(p, num_samples=1).item()

    def predict(self, prompt: str, num_preds: int, temperature=1.0):
        """Simulate character generation one at a time."""

        # Iterate over warmup text. RNN cell outputs final state
        warmup_indices = self.tokenizer.encode(prompt.lower()).tolist()
        outs, state = self.model(self._inp(warmup_indices), return_state=True)

        # Sample next token and update state
        indices = []
        for _ in range(num_preds):
            i = self.sample_token(outs[-1], temperature)
            indices.append(i)
            outs, state = self.model(self._inp([i]), state, return_state=True)

        return self.tokenizer.decode(warmup_indices + indices)

**Sanity test.** Completing 'thank you':

In [ ]:
textgen = TextGenerator(model, tokenizer, device="cpu")
s = [textgen.predict("thank y", num_preds=2, temperature=0.4) for i in range(20)]
(np.array(s) == "thank you").mean()

**Example.** The network can generate output given warmup prompt of arbitrary length. Here we also look at the effect of temperature on the generated text:

In [ ]:
warmup = "mr williams i underst"
text = []
temperature = []
for i in range(1, 6):
    t = 0.20 * i
    s = textgen.predict(warmup, num_preds=100, temperature=t)
    text.append(s)
    temperature.append(t)

In [ ]:
from IPython.display import display
import pandas as pd

pd.set_option("display.max_colwidth", None)
df = pd.DataFrame({"temp": [f"{t:.1f}" for t in temperature], "text": text})
df = df.style.set_properties(**{"text-align": "left"})
display(df)

The generated text appear more random as we increase the sampling temperature[^softmax-approx]. Conversely, as the temperature decreases, the softmax function behaves more like an argmax. In this scenario, the sampling algorithm selects the token with the highest probability, which increases the likelihood of cycles.

[^softmax-approx]: That is, $e^s \approx 1 + x$ for $|x| \ll 1$, so that $p_k = \frac{e^{s_k}}{\sum_j e^{s_j}} \approx \frac{1 + s_k}{K + \sum_j s_j}.$ 

# Backprop Through Time (BPTT)

Recall that the state is updated at each time step with the current input. Thus, we have to track the dependencies across time steps where the RNN parameters are shared. This is called *Backprogation Through Time* (BPTT) or BP for sequence models.
Hopefully, this discussion will bring some precision to the notion of vanishing and exploding gradients. 

This procedure requires us to expand (or unroll) the computational graph of an RNN one time step at a time. The unrolled RNN is essentially a feedforward neural network with the special property that the same parameters are repeated throughout the unrolled network, appearing at each time step.
Then, we can apply the usual BP through the unrolled net. In particular, we want to see causality in the equations, i.e. state at time $t$ only influences future time steps.

For long sequences, e.g. text sequences containing over a thousand tokens, BP across many layers poses problems both from a computational (too much memory to compress in a single state vector) and optimization standpoint (numerical instability). Here input from the first step passes through $T$ matrix products before arriving at the output. Similarly, we expect $T$ matrix products are required to compute the gradient at the first time step.

![RNN cell backpropation. Note that the matrices $\mathbf{W}, \mathbf{U},$ and $\mathbf{V}$ are shared across time steps.](img/04-rnn-backprop.svg){#fig-04-rnn-backprop fig-align="center" width=500px}

Recall:

$$
\begin{aligned}
\mathbf{H}_t &= f(\mathbf{X}_t \mathbf{U} + \mathbf{H}_{t-1} \mathbf{W} + \mathbf{b}) \\
\mathbf{Y}_t &= \mathbf{H}_t \mathbf{V} + \mathbf{c} \\
\mathbf{H}_{t+1} &= f(\mathbf{X}_{t+1} \mathbf{U} + \mathbf{H}_{t} \mathbf{W} + \mathbf{b}).
\end{aligned}
$$

Assume incoming gradients $\frac{\partial \mathcal{L}}{\partial \mathbf{Y}_t}$ and $\frac{\partial \mathcal{L}}{\partial \mathbf{H}_{t+1}}$ from the next layer.
We start by calculating the gradient with respect to $\mathbf{V}.$ Here we abstract the product between two tensors on appropriate indices by using the $\text{prod}$ notation. The exact formula can be recovered with [tensor index notation](https://en.wikipedia.org/wiki/Einstein_notation). Let $f$ be an activation function. Upper case indicates that a tensor's first dimension is the batch dimension when applicable. Then, 

$$
\begin{aligned}
\underbrace{\frac{\partial \mathcal{L}}{\partial \mathbf{V}}}_{(h, q)} &= \sum_{t=1}^T \text{prod}\left(\frac{\partial \mathcal{L}}{\partial \mathbf{Y}_t}, \frac{\partial \mathbf{Y}_t}{\partial \mathbf{V}}\right) = \sum_{t=1}^T \underbrace{\mathbf{H}_t^\top \frac{\partial \mathcal{L}}{\partial \mathbf{Y}_t}}_{(h, B) \,\times\, (B, q)} \\
\underbrace{\frac{\partial \mathcal{L}}{\partial \mathbf{c}}}_{(1, q)}
&= \sum_{t=1}^T \text{prod}\left(\frac{\partial \mathcal{L}}{\partial \mathbf{Y}_t}, \frac{\partial \mathbf{Y}_t}{\partial \mathbf{c}}\right) = \sum_{t=1}^T \underbrace{\mathbf{1}^\top \frac{\partial \mathcal{L}}{\partial \mathbf{Y}_t}}_{(1, B) \,\times\, (B, q)} 
\end{aligned}
$$

Next, we calculate the gradients flowing to $\mathbf{H}_t$ which will be our gateway to compute gradients of $\mathbf{W}$, $\mathbf{U}$, and $\mathbf{b}$, and finally $\mathbf{X}_t.$ Note that $\mathbf{H}_t$ affects not only $\mathbf{Y}_t$, but also future $\mathbf{Y}_{t^\prime}$ via $\mathbf{H}_{t^\prime}$ for $t^\prime > t.$ But in terms of direct dependence, the nodes that immediately depend on $\mathbf{H}_t$ are $\mathbf{Y}_t$ and $\mathbf{H}_{t+1}$ (@fig-04-rnn-backprop). Let $\mathbf{Z}_{t+1} = \mathbf{X}_{t+1} \mathbf{U} + \mathbf{H}_{t} \mathbf{W} + \mathbf{b}.$ Then,

$$
\begin{aligned}
\underbrace{\frac{\partial\mathcal{L}}{\partial\mathbf{H}_t}}_{(B, h)}
&= 
\text{prod}\left(
    \frac{\partial\mathcal{L}}{\partial\mathbf{Y}_t}, 
    \frac{\partial\mathbf{Y}_t}{\partial\mathbf{H}_t}
\right) + 
\text{prod}\left(
    \frac{\partial\mathcal{L}}{\partial\mathbf{H}_{t + 1}}, 
    \frac{\partial\mathbf{H}_{t + 1}}{\partial\mathbf{Z}_{t+1}},
    \frac{\partial\mathbf{Z}_{t+1}}{\partial\mathbf{H}_{t }}
\right) \\
&= 
\underbrace{\frac{\partial\mathcal{L}}{\partial\mathbf{Y}_t}\, \mathbf{V}^\top}_{(B, q)\,\times\,(q, h)} +
\underbrace{
    \left(
        \frac{\partial\mathcal{L}}{\partial\mathbf{H}_{t + 1}}
        \odot 
        f^\prime(\mathbf{Z}_{t+1})
    \right) \mathbf{W}^\top
}_{((B, h)\, \cdot \, (B, h)) \, \times \, (h, h)}
\end{aligned}
$$ {#eq-state_vec_grad}

To make sense of this, recall $\mathbf{V}$ and $\mathbf{W}$ acts on $\mathbf{H}_t$ from the left. Hence, when we take its transpose, multiplying a tensor to the right of $\frac{\partial\mathcal{L}}{\partial\mathbf{H}_t}$, results in a summation along the dimension containing information about the state $\mathbf{H}_t.$ 
Similarly, the orientation of the products within the expression are also correct.

Note that the above expression is recursive, we should be able to get a closed form expression from terms in time step $t, t+1, \ldots, T.$ For tractability, let's assume we have no nonlinearity, or $f = \text{Id},$ so that $f^\prime(\mathbf{Z}_{t + 1}) = \mathbf{1}_{(B, h)}$. Then, we can write:

$$
\begin{aligned}
a_t = \frac{\partial\mathcal{L}}{\partial\mathbf{H}_t}, \quad
b_t = \frac{\partial\mathcal{L}}{\partial\mathbf{Y}_t}\, \mathbf{V}^\top, \quad
c_t = c = \mathbf{W}^\top
\end{aligned}
$$

with $a_{T+1} = 0$ and $a_T = b_T.$ Hence,

$$
\begin{aligned}
a_t &= b_t + a_{t+1} c_t \\
&= b_t + (b_{t + 1} + a_{t + 2} c_{t + 1}) c_t \\ 
&= b_t + (b_{t + 1} + (b_{t + 2} + a_{t + 3} c_{t + 2}) c_{t + 1}) c_t \\
&= b_t + b_{t + 1} c_t + b_{t + 2}c_{t + 1}c_t + a_{t + 3} c_{t + 2}c_{t + 1}c_t \\
&\vdots \\
&= \sum_{\kappa = 0}^{T - t} b_{t + \kappa}\, c^{\kappa}.
\end{aligned}
$$

Thus,

$$
\boxed{
\frac{\partial\mathcal{L}}{\partial\mathbf{H}_t} = 
 \sum_{\kappa = 0}^{T - t}
\frac{\partial\mathcal{L}}{\partial\mathbf{Y}_{t + \kappa}}\, \mathbf{V}^\top
\left(\mathbf{W}^\top\right)^{\kappa}.
}
$$ (state_vec_grad)

<br>

This formula is similar to that for gradient flow across the layers of a deep MLP network, but here the depth is along sequence length. 
The terms in the sum correspond to paths of increasing *path lengths* $\kappa = 0, \ldots, T - t$ from the  current time step $t.$ Finally, observe that the change in loss due to the current time step is only due to its effect on future time steps, not on the past, so we have a notion of causality in RNNs.

<br>

![Gradient transformation graph to get $\frac{\partial\mathcal{L}}{\partial\mathbf{H}_t}$ at time step $t$ with increasing path length $\kappa.$ Each edge is modulated by $f^\prime$ and $\mathbf{W}^\top.$](img/04-rnn-bptt.svg){#fig-04-rnn-bptt fig-align="center" width=600px}

Finally, let's calculate the rest of the parameter gradients. Then,

$$
\begin{aligned}
\underbrace{\frac{\partial \mathcal{L}}{\partial \mathbf{U}}}_{(d, h)} &= \sum_{t=1}^T \text{prod}\left(
    \frac{\partial \mathcal{L}}{\partial \mathbf{H}_t}, 
    \frac{\partial \mathbf{H}_t}{\partial \mathbf{Z}_t},
    \frac{\partial \mathbf{Z}_t}{\partial \mathbf{U}}
\right) 
= 
\sum_{t=1}^T \underbrace{
    \mathbf{X}_{t}^\top 
    \left(
        \frac{\partial\mathcal{L}}{\partial\mathbf{H}_{t}}
        \odot 
        f^\prime(\mathbf{Z}_{t})
    \right)
}_{(d, B) \,\times\, ((B, h) \, \cdot\, (B, h))}
\\\\
\underbrace{\frac{\partial \mathcal{L}}{\partial \mathbf{W}}}_{(h, h)} &= \sum_{t=1}^T \text{prod}\left(
    \frac{\partial \mathcal{L}}{\partial \mathbf{H}_t}, 
    \frac{\partial \mathbf{H}_t}{\partial \mathbf{Z}_t},
    \frac{\partial \mathbf{Z}_t}{\partial \mathbf{W}}
\right) 
= 
\sum_{t=1}^T \underbrace{\mathbf{H}_{t-1}^\top \left(
        \frac{\partial\mathcal{L}}{\partial\mathbf{H}_{t}}
        \odot 
        f^\prime(\mathbf{Z}_{t})
    \right)}_{(h, B) \,\times\, ((B, h) \, \cdot\, (B, h))} 
\\\\
\underbrace{\frac{\partial \mathcal{L}}{\partial \mathbf{b}}}_{(1, h)}
&= 
\sum_{t=1}^T \text{prod}\left(
    \frac{\partial \mathcal{L}}{\partial \mathbf{H}_t}, 
    \frac{\partial \mathbf{H}_t}{\partial \mathbf{Z}_t},
    \frac{\partial \mathbf{Z}_t}{\partial \mathbf{b}}
\right) 
= \sum_{t=1}^T \underbrace{\mathbf{1}^\top
    \left(
        \frac{\partial\mathcal{L}}{\partial\mathbf{H}_{t}}
        \odot 
        f^\prime(\mathbf{Z}_{t})
    \right)
}_{(1, B) \,\times\, ((B, h) \, \cdot \, (B, h))}.
\end{aligned}
$$

The gradient to inputs may be also relevant (e.g. deep RNNs):

$$
\underbrace{\frac{\partial \mathcal{L}}{\partial \mathbf{X}_t}}_{(B, d)}
=
\text{prod}\left(
    \frac{\partial \mathcal{L}}{\partial \mathbf{H}_t}, 
    \frac{\partial \mathbf{H}_t}{\partial \mathbf{Z}_t},
    \frac{\partial \mathbf{Z}_t}{\partial \mathbf{X}_t}
\right) 
= \underbrace{
    \left(
        \frac{\partial\mathcal{L}}{\partial\mathbf{H}_{t}}
        \odot 
        f^\prime(\mathbf{Z}_{t})
    \right) \mathbf{U}^\top
}_{((B, h) \, \cdot \, (B, h)) \, \times \, (h, d)}
$$

Hence, the key quantity that affects the numerical stability is $\frac{\partial\mathcal{L}}{\partial\mathbf{H}_t}$ @eq-state_vec_grad. 

<br>

## Manual verification

In [ ]:
import torch
import torch.nn.functional as F

B, T, V, h = 32, 5, 30, 64

# forward pass
O = torch.randint(low=0, high=V, size=(T, B))    # (T, B)
x = torch.randint(low=0, high=V, size=(T, B))    # (T, B)
X = F.one_hot(x, num_classes=V).float()          # (T, B, V)
X.requires_grad = True

model = LanguageModel(RNN)(V, h, V)   

W  = model.cell.W                                # (h, h)
U  = model.cell.U                                # (V, h)
b  = model.cell.b                                # (h,)
Vt = model.linear.weight                         # (V, h)
c  = model.linear.bias                           # (V,)
Y  = model(X)                                    # (T, B, V)
H  = model.cell(X)[0]                            # (T, B, h)
J  = 1 - H * H                                   # (T, B, h)

# backprop
X.retain_grad()
Y.retain_grad()
loss = F.cross_entropy(Y.transpose(1, 2), O)
loss.backward(retain_graph=True)

Smoke test:

In [ ]:
assert ((H @ Vt.T + c) - Y).abs().max() == 0.0

Calculating the gradients by hand: 

In [ ]:
dY = Y.grad
dH = [None] * T
dH[T - 1] = dY[T - 1] @ Vt
for t in range(T - 2, -1, -1):
    dH[t] = dY[t] @ Vt + (dH[t + 1] * J[t + 1]) @ W.T
    
dH = torch.stack(dH)
dZ = dH * J
dc = torch.einsum('tbj -> j', dY)
dV = torch.einsum('tbh, tbv -> hv', H, dY)
dU = torch.einsum('tbv, tbh -> vh', X, dH * J)
db = torch.einsum('tbh -> h', dH * J)
dX = torch.einsum('tbh, vh -> tbv', dH * J, U)
dW = sum([H[t-1].T @ (dH * J)[t] for t in range(1, T)], torch.zeros((h, h)))

Calculating absolute errors versus `autograd`:

In [ ]:
def compare(name, dt, t):
    exact = torch.all(dt == t.grad).item()
    approx = torch.allclose(dt, t.grad, rtol=1e-5)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{name:<3s} | exact: {str(exact):5s} | approx: {str(approx):5s} | maxdiff: {maxdiff:.2e}')
    return approx

assert compare('dV', dV.T, Vt)
assert compare('dc', dc, c)
assert compare('dU', dU, U)
assert compare('dW', dW, W)
assert compare('db', db, b)
assert compare('dX', dX, X)

# Modern RNN architectures

Recall that the key quantity to calculate RNN gradients is $\frac{\partial \mathcal{L}}{\partial \mathbf{H}_t}$ that contain terms $\left(\mathbf{W}^\top\right)^{\kappa}$ for path length $\kappa$ from the current time step $t,$ where $\mathbf{W}$ is the matrix responsible for transforming the latent state vector. This explodes or vanishes with increasing $\kappa$ depending on whether the norm of the principal eigenvalue of $\mathbf{W}$ is greater than or equal to 1. Hence, RNNs are said to have problems with modeling long-term dependencies between tokens. The consequence of this in practice is that RNNs are limited in context size. 

Exploding gradients can be solved in practice by gradient clipping or truncated BPTT. On the other hand, the problem of vanishing gradients requires nontrivial architectural changes. We will consider **LSTM** [@lstm], **GRU** [@gru], as well as **deep RNNs** and **Bidirectional RNNs** [@birnn] which increase model complexity. In the next chapter, we will apply these architectures to sequence-to-sequence tasks.

**Code.** In terms of code, we implement the following classes in order: `RNN`, `LSTM`, and `GRU`, as well as wrappers `Deep` and `Bidirectional` that augment units to the respective architecture, but with the same API. Note that we can also swap with PyTorch implementations, e.g.

```python
Deep(Bidirectional(nn.LSTM))(EMBED_DIM, HIDDEN_DIM, num_layers=3, batch_first=True)
```

Finally, all implementations are checked for correctness by comparing with PyTorch. 🔥

# Long Short-Term Memory (LSTM)

LSTMs [@lstm] (1997) are one of first successful approaches for dealing with vanishing gradients. An LSTM cell resembles an RNN unit but with two state vectors maintained for an input sequence $\mathbf{X}_t$: 
the *internal state* $\mathbf{C}_t$, which change slowly during training, represent the long-term memory of the cell, while the *hidden state* $\mathbf{H}_t$ (analogous to RNN's) which are ephemeral activations passed to the next layer, represent short-term memory. 

Below we will that these two state vectors occupy the two main compute streams inside an LSTM cell, with $\mathbf{C}_t$ acting like a "highway" for gradients, allowing them to flow with minimal obstruction during backpropagation. This reduces the risk of vanishing gradients and helps the model maintain long-term dependencies. On the other hand $\mathbf{H}_t$ is essentially long-term memory augmented with recent inputs to make current decisions realized  by the **gating mechanism**.

## Gating mechanism

LSTM uses gates to control the flow of information over time. 
More precisely, the gates implement dedicated mechanisms for updating and resetting internal states given current input. The ff. are the gates in an LSTM cell:

| Gate | Symbol | Controls |
| :--: | :--: | :--: |
| Input | $\boldsymbol{\Gamma}^i$  | How a given input should impact the internal state |
| Forget | $\boldsymbol{\Gamma}^f$  |  How much of the previous internal state is *retained*  | 
| Output | $\boldsymbol{\Gamma}^o$  | How much does the current internal state affect the output | 

The LSTM cell has an internal state $\mathbf{c}_t$, interpreted as the long-term memory, as opposed to the hidden state $\mathbf{h}_t$ which is interpreted as short-term memory. The latter vector is what is passed as the output of the cell at each time step.

## LSTM equations

The LSTM equations are as follows. Let $h$ be the width of the LSTM cell and $B$ be the batch size. Then all entries of the ff. tensors are in $(0, 1) \subset \mathbb{R}$ which can be interpreted as a $B \times h$ smooth switch:

$$
\begin{aligned}
{\boldsymbol{\Gamma}^i_t} & =\sigma(\mathbf{X}_t \mathbf{U}^{{i}}+\mathbf{H}_{t-1} \mathbf{W}^{{i}}+\mathbf{b}^{{i}}) \\
{\boldsymbol{\Gamma}^f_t} & =\sigma(\mathbf{X}_t \mathbf{U}^{{f}}+\mathbf{H}_{t-1} \mathbf{W}^{{f}}+\mathbf{b}^{{f}}) \\
{\boldsymbol{\Gamma}^o_t} & =\sigma(\mathbf{X}_t \mathbf{U}^{{o}}+\mathbf{H}_{t-1} \mathbf{W}^{{o}}+\mathbf{b}^{{o}})
\end{aligned}
$$

The internal cell state ${\mathbf{C}}_t$ with shape $B \times h$ is calculated as

$$
\begin{aligned}
\tilde{\mathbf{C}}_t &= \tanh(\mathbf{X}_t \mathbf{U}^{{c}}+\mathbf{H}_{t-1} \mathbf{W}^{{c}}+\mathbf{b}^{{c}}) \\
\mathbf{C}_t &= {\boldsymbol{\Gamma}^f_t} \odot \mathbf{C}_{t - 1} + {\boldsymbol{\Gamma}^i_t} \odot \tilde{\mathbf{C}}_t
\end{aligned}
$$

where $\tilde{\mathbf{C}}_t$ is called the *candidate state*. Note that the candidate state is computed similarly to the gates, except that it has $\tanh$ activations so that its values are in the more dynamic range $(-1, 1).$ Finally, the hidden state is defined as:

$$
\begin{aligned}
{\mathbf{H}}_t &= {\boldsymbol{\Gamma}^o_t} \odot \tanh({\mathbf{C}}_t).
\end{aligned}
$$

Applying $\tanh$ ensures that elements of ${\mathbf{H}}_t$ are in $(-1, 1).$ The computation is illustrated in @fig-05-lstm:

<br>

![LSTM cell computation. Notice the "highway" for the internal state $\mathbf{C}_t.$](img/05-lstm.svg){#fig-05-lstm fig-align="center" width=600px}

If ${\boldsymbol{\Gamma}^f_t} = 1$ and ${\boldsymbol{\Gamma}^i_t} = 0$, then the internal state is constant, i.e. $\mathbf{c}_{t-1} = \mathbf{c}_t.$ For example, when the current input $\mathbf{x}_t$ is not important. On the other hand the internal state is reset when ${\boldsymbol{\Gamma}^f_t} = 0.$
In general, the input and forget gates give the model enough flexibility to maintain or update the internal state in response to subsequent inputs. In practice, this minimizes vanishing gradients since the internal state does not pass through a linear layer. That is, the gradient 
along $\frac{\partial{\mathbf{c}_t}}{\partial{\mathbf{c}_{t-1}}}$ only passses through ${\boldsymbol{\Gamma}^f_t} \, \odot$ which is fair. The exact gating dynamics are learned during training.

For the hidden state, the output gate controls what of the current internal state to impact the next layers. For example, ${\boldsymbol{\Gamma}^o_t} = 0$ prevents the current memory from affecting the gating in the next layer, as well as getting zero output at the current step. This can happen across many time steps, where the internal state accrues information across many time steps, and then suddenly impact the network as soon as the output gate flips to values close to 1. This demonstrates the richness of the dynamics of the LSTM cell. 

**Remark.** We can now see why the hidden state $\mathbf{h}_t$ is interpreted as short-term memory since it integrates tightly with the current input at the current step, and is derived, based on the current input via ${\boldsymbol{\Gamma}^o_t}$, from the long-term memory $\mathbf{c}_t$ for the next step. The internal state $\mathbf{c}_t$, on the other hand, is able to accrue and update information over many time steps depending on the exact gating events.

<br>

## Code implementation

Observe that LSTM cell and language model has the same API as the RNN: 

In [ ]:
class LSTM(RNNBase):
    def __init__(self, inputs_dim: int, hidden_dim: int):
        super().__init__(inputs_dim, hidden_dim)
        self.I = nn.Linear(inputs_dim + hidden_dim, hidden_dim)
        self.F = nn.Linear(inputs_dim + hidden_dim, hidden_dim)
        self.O = nn.Linear(inputs_dim + hidden_dim, hidden_dim)
        self.G = nn.Linear(inputs_dim + hidden_dim, hidden_dim)

    def init_state(self, x):
        B = x.shape[1]
        h = torch.zeros(B, self.hidden_dim, device=x.device)
        c = torch.zeros(B, self.hidden_dim, device=x.device)
        return h, c
    
    def _step(self, x_t, state):
        h, c = state
        x_gate = torch.cat([x_t, h], dim=1)
        g = torch.tanh(self.G(x_gate))
        i = torch.sigmoid(self.I(x_gate))
        f = torch.sigmoid(self.F(x_gate))
        o = torch.sigmoid(self.O(x_gate))
        c = f * c + i * g
        h = o * torch.tanh(c)
        return h, (h, c)

    def compute(self, x, state):
        T = x.shape[0]
        outs = []
        for t in range(T):
            out, state = self._step(x[t], state)
            outs.append(out)
        return torch.stack(outs), state

**Remark.** For efficient processing the weights of the linear layers can be fused:

$$
\begin{aligned}
\mathbf{G} = \mathbf{X}_t [\mathbf{U}^{{i}}| \mathbf{U}^{{f}}| \mathbf{U}^{{o}}
| \mathbf{U}^{{c}}]+\mathbf{H}_{t-1} [\mathbf{W}^{{i}}| \mathbf{W}^{{f}}| \mathbf{W}^{{o}} | \mathbf{W}^{{c}}]+ [\mathbf{b}^{{i}} \oplus \mathbf{b}^{{f}} \oplus \mathbf{b}^{{o}} \oplus \mathbf{b}^{{c}}].
\end{aligned}
$$

Then $\mathbf{G}$ is later sliced to apply the activations.

Shapes test:

In [ ]:
B, T, d, h = 32, 5, 10, 20
x = torch.randn(T, B, d)
lstm = LSTM(d, h)
outs, (H, C) = lstm(x)  # same with PyTorch: 
                        # https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html

assert outs.shape == (T, B, h)
assert H.shape == (B, h)
assert C.shape == (B, h)

Correctness test:

In [ ]:
lstm_torch = nn.LSTM(d, h)
for net in [lstm, lstm_torch]:
    for name, p in net.named_parameters():
        if "bias" in name:
            p.data.fill_(0.0)
        else:
            p.data.fill_(1.0)

error = torch.abs(lstm(x)[0] - lstm_torch(x)[0]).max()
print(error)
assert error < 1e-6

<br>

### Model training

In [ ]:
from torch.utils.data import random_split

data, tokenizer = TimeMachine().build()
T = 30
BATCH_SIZE = 128
VOCAB_SIZE = tokenizer.vocab_size

dataset = SequenceDataset(data, seq_len=T, vocab_size=VOCAB_SIZE)
train_dataset, valid_dataset = random_split(dataset, [0.80, 0.20])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)    # also sampled

print("preds per epoch")
print("train:", f"{len(train_loader) * BATCH_SIZE * T: .3e}")
print("valid:", f"{len(valid_loader) * BATCH_SIZE * T: .3e}")

LSTM language model just replaces RNN cell with LSTM in `LanguageModel` wrapper:

In [ ]:
from tqdm.notebook import tqdm

DEVICE = "cpu"
LR = 0.01
EPOCHS = 5
MAX_NORM = 1.0

model = LanguageModel(LSTM)(VOCAB_SIZE, 64, VOCAB_SIZE)
model.to(DEVICE)
optim = torch.optim.Adam(model.parameters(), lr=LR)

train_losses = []
valid_losses = []
for e in tqdm(range(EPOCHS)):
    for t, (x, y) in tqdm(enumerate(train_loader), total=len(train_loader)):
        x, y = x.to(DEVICE), y.to(DEVICE)
        loss = train_step(model, optim, x, y, MAX_NORM)
        train_losses.append(loss)

        if t % 5 == 0:
            xv, yv = next(iter(valid_loader))
            xv, yv = xv.to(DEVICE), yv.to(DEVICE)
            valid_losses.append(valid_step(model, xv, yv))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats("svg")

plt.figure(figsize=(10, 4))
plt.plot(train_losses, label="train")
plt.plot(np.array(range(1, len(valid_losses) + 1)) * 5, valid_losses, label="valid")
plt.grid(linestyle="dotted", alpha=0.6)
plt.ylabel("loss")
plt.xlabel("step")
plt.legend();

In [ ]:
np.array(valid_losses[-50:]).mean()

<br>

### Text generation

In [ ]:
textgen = TextGenerator(model, tokenizer, device="cpu")
s = [textgen.predict("thank y", num_preds=2, temperature=0.4) for i in range(20)]
(np.array(s) == "thank you").mean()

In [ ]:
warmup = "mr williams i underst"
text = []
temperature = []
for i in range(1, 6):
    t = 0.20 * i
    s = textgen.predict(warmup, num_preds=100, temperature=t)
    text.append(s)
    temperature.append(t)

In [ ]:
import pandas as pd
from IPython.display import display
pd.set_option("display.max_colwidth", None)
df = pd.DataFrame({"temp": [f"{t:.1f}" for t in temperature], "text": text})
df = df.style.set_properties(**{"text-align": "left"})
display(df)

# Gated Recurrent Unit (GRU)

Recall that the hidden state in the LSTM cell is essentially the internal state filtered based on the current input (i.e. with the output gate $\boldsymbol{\Gamma}^o_t$). 
Modifications such as [peephole connections](https://static.googleusercontent.com/media/research.google.com/en//pubs/archive/43905.pdf) have been proposed to include the pure internal cell state in the computation. A further simplification is the **GRU** [@gru] which combines the feedback mechanism into a single hidden state vector that is appropriately gated. 
This significantly reduces the number of parameters compared to LSTM with similar performance.

## Gating mechanism

Similar to an LSTM cell, GRU uses gates to control update and reset of memory over time:

| Gate | Symbol | Controls | 
| :--: | :--: | :--: |
| Reset | $\boldsymbol{\Gamma}^r$  | How much of the previous state is incorporated into the new state | 
| Update | $\boldsymbol{\Gamma}^z$  |  How much of the old state is *retained*, versus the candidate state |


This is similar to the LSTM cell with the input gate corresponding to the reset gate and the forget gate corresponding to the update gate. There is no output gate; the current state is calculated as a weighted average of the previous state and the candidate state. 

## GRU equations

GRU computation is defined as follows. Let $h$ be the width of the GRU and $B$ be the batch size. Then all entries of the gates in $(0, 1) \subset \mathbb{R}$ which can be interpreted as a $B \times h$ smooth switch:

$$
\begin{aligned}
{\boldsymbol{\Gamma}^r_t} & =\sigma(\mathbf{X}_t \mathbf{U}^{{r}}+\mathbf{H}_{t-1} \mathbf{W}^{{r}}+\mathbf{b}^{{r}}) \\
{\boldsymbol{\Gamma}^z_t} & =\sigma(\mathbf{X}_t \mathbf{U}^{{z}}+\mathbf{H}_{t-1} \mathbf{W}^{{z}}+\mathbf{b}^{{z}}) 
\end{aligned}
$$

The hidden state ${\mathbf{C}}_t$ with shape $B \times h$ is updated as

$$
\begin{aligned}
\tilde{\mathbf{H}}_t &= \tanh(\mathbf{X}_t \mathbf{U}^{{h}}+(\boldsymbol{\Gamma}^r_t \odot \mathbf{H}_{t-1}) \mathbf{W}^{{h}}+\mathbf{b}^{{h}}) \\
\mathbf{H}_t &= {\boldsymbol{\Gamma}^z_t} \odot \mathbf{H}_{t - 1} + (1 - {\boldsymbol{\Gamma}^z_t}) \odot \tilde{\mathbf{H}}_t
\end{aligned}
$$

where $\tilde{\mathbf{H}}_t$ is called the *candidate hidden state*. Applying $\tanh$ ensures that elements of ${\mathbf{H}}_t$ are in $(-1, 1).$  The computation is illustrated in @fig-05-gru:

<br>

![Computing the hidden state in a GRU model. [Source](https://www.d2l.ai/chapter_recurrent-modern/gru.html#hidden-state)](img/05-gru.svg){#fig-05-gru fig-align="center" width=600px}

First, let us look at the reset gate. When ${\boldsymbol{\Gamma}^r_t} = 1$, the entire hidden state is used to calculate the candidate hidden state. On the other hand, if ${\boldsymbol{\Gamma}^r_t} = 0$, then the hidden state is reset and we start with a candidate state that depends precisely on the current input. However, the update gate can still ignore the candidate state with ${\boldsymbol{\Gamma}^z_t} = 1$, so that past inputs will potentially still have an effect on future outputs, i.e. $\mathbf{h}_t \approx \mathbf{h}_{t - 1}.$  Similarly, the GRU can accrue information across many time steps this way by keeping the reset and update gates open. Once it closes, the accumulated information in the candidate state takes effect. In general, this is how long-term dependencies are handled by the unit. The gradient for $\mathbf{h}_{t-1}$ can be calculated by following nodes that depend on it, there are four such paths, but one path 
involving $\frac{\partial{\mathbf{h}_t}}{\partial{\mathbf{h}_{t-1}}}$ only involves scaling with ${\boldsymbol{\Gamma}^z_t} \, \odot$ instead of matrix multiplication.

<br>

## Code implementation

In [ ]:
class GRU(RNNBase):
    def __init__(self, inputs_dim: int, hidden_dim: int):
        super().__init__(inputs_dim, hidden_dim)
        self.hidden_dim = hidden_dim
        self.inputs_dim = inputs_dim
        self.R = nn.Linear(inputs_dim + hidden_dim, hidden_dim)
        self.Z = nn.Linear(inputs_dim + hidden_dim, hidden_dim)
        self.G = nn.Linear(inputs_dim + hidden_dim, hidden_dim)
    
    def init_state(self, x):
        B = x.shape[1]
        return torch.zeros(B, self.hidden_dim, device=x.device)

    def _step(self, x_t, state):
        h = state
        x_gate = torch.cat([x_t, h], dim=1)
        r = torch.sigmoid(self.R(x_gate))
        z = torch.sigmoid(self.Z(x_gate))
        g = torch.tanh(self.G(torch.cat([x_t, r * h], dim=1)))
        h = z * h + (1 - z) * g
        return h, h

    def compute(self, x, state):
        T = x.shape[0]
        outs = []
        for t in range(T):
            out, state = self._step(x[t], state)
            outs.append(out)
        return torch.stack(outs), state

Shapes test:

In [ ]:
B, T, d, h = 32, 5, 10, 20
x = torch.randn(T, B, d)
gru = GRU(d, h)
outs, H = gru(x)    # same with PyTorch: 
                    # https://pytorch.org/docs/stable/generated/torch.nn.GRU.html

assert outs.shape == (T, B, h)
assert H.shape == (B, h)
assert all(H[0] == outs[-1][0])

<br>

### Model training

In [ ]:
from torch.utils.data import random_split

data, tokenizer = TimeMachine().build()
T = 30
BATCH_SIZE = 128
VOCAB_SIZE = tokenizer.vocab_size

dataset = SequenceDataset(data, seq_len=T, vocab_size=VOCAB_SIZE)
train_dataset, valid_dataset = random_split(dataset, [0.80, 0.20])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)    # also sampled

print("preds per epoch")
print("train:", f"{len(train_loader) * BATCH_SIZE * T: .3e}")
print("valid:", f"{len(valid_loader) * BATCH_SIZE * T: .3e}")

In [ ]:
from tqdm.notebook import tqdm

DEVICE = "cpu"
LR = 0.01
EPOCHS = 5
MAX_NORM = 1.0

model = LanguageModel(GRU)(VOCAB_SIZE, 64, VOCAB_SIZE)
model.to(DEVICE)
optim = torch.optim.Adam(model.parameters(), lr=LR)

train_losses = []
valid_losses = []
for e in tqdm(range(EPOCHS)):
    for t, (x, y) in tqdm(enumerate(train_loader), total=len(train_loader)):
        x, y = x.to(DEVICE), y.to(DEVICE)
        loss = train_step(model, optim, x, y, MAX_NORM)
        train_losses.append(loss)

        if t % 5 == 0:
            xv, yv = next(iter(valid_loader))
            xv, yv = xv.to(DEVICE), yv.to(DEVICE)
            valid_losses.append(valid_step(model, xv, yv))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats("svg")

plt.figure(figsize=(10, 4))
plt.plot(train_losses, label="train")
plt.plot(np.array(range(1, len(valid_losses) + 1)) * 5, valid_losses, label="valid")
plt.grid(linestyle="dotted", alpha=0.6)
plt.ylabel("loss")
plt.xlabel("step")
plt.legend();

In [ ]:
np.array(valid_losses[-50:]).mean()

<br>

### Text generation

In [ ]:
textgen = TextGenerator(model, tokenizer, device="cpu")
s = [textgen.predict("thank y", num_preds=2, temperature=0.4) for i in range(20)]
(np.array(s) == "thank you").mean()

In [ ]:
warmup = "mr williams i underst"
text = []
temperature = []
for i in range(1, 6):
    t = 0.20 * i
    s = textgen.predict(warmup, num_preds=100, temperature=t)
    text.append(s)
    temperature.append(t)

In [ ]:
import pandas as pd
from IPython.display import display
pd.set_option("display.max_colwidth", None)
df = pd.DataFrame({"temp": [f"{t:.1f}" for t in temperature], "text": text})
df = df.style.set_properties(**{"text-align": "left"})
display(df)

# Deep Recurrent Networks

To increase network complexity, we can make the RNNs "vertically deep" at each time step (i.e. deep in the usual FFN sense). Recall that RNNs are "horizontally deep" in the sense that early inputs at influence outputs and state at later time steps. But having depth in the usual sense allows learning *higher-order* state vectors $\mathbf{H}^{\ell}_t.$

<br>

![Deep RNN architecture. Observe that it requires $L$ state vectors at each step.](img/05-deep-rnn.svg){#fig-05-deep-rnn fig-align="center" width=330px}

<br>

## Code implementation

In [ ]:
from torch.utils.data import random_split

data, tokenizer = TimeMachine().build()
T = 30
BATCH_SIZE = 128
VOCAB_SIZE = tokenizer.vocab_size

dataset = SequenceDataset(data, seq_len=T, vocab_size=VOCAB_SIZE)
train_dataset, valid_dataset = random_split(dataset, [0.80, 0.20])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)    # also sampled

print("preds per epoch")
print("train:", f"{len(train_loader) * BATCH_SIZE * T: .3e}")
print("valid:", f"{len(valid_loader) * BATCH_SIZE * T: .3e}")

**Deep RNN**. Note that at each time step, we compute $\mathbf{o}_t^\ell, \mathbf{h}_t^\ell = f(\mathbf{o}_t^{\ell - 1}, \mathbf{h}_{t-1}^\ell)$ where $\mathbf{o}_t^0 = \mathbf{x}_t.$ This can be visualized as the output vector moving depthwise, while we feed the previous states horizontally as in @fig-05-deep-rnn. Hence, we can calculate all outputs at the same depth, before pushing it to the next layer (i.e. fix $\ell$ and iterate over $t$).

In [ ]:
from functools import partial

class DeepRNN(RNNBase):
    def __init__(self, 
        cell: Type[RNNBase],
        inputs_dim: int, hidden_dim: int,
        num_layers: int,    # (!)
        **kwargs,
    ):
        super().__init__(inputs_dim, hidden_dim)
        self.num_layers = num_layers
        self.layers = nn.ModuleList()
        for l in range(num_layers):
            if l == 0:
                self.layers.append(cell(inputs_dim, hidden_dim, **kwargs))
            else:
                self.layers.append(cell(hidden_dim, hidden_dim, **kwargs))
  
    def init_state(self, x):
        """Defer state init to each cell with state=None."""
        return [None] * self.num_layers
    
    def compute(self, x, state):
        T = x.shape[0]
        out = x
        for l, cell in enumerate(self.layers):
            out, state[l] = cell(out, state[l])
        return out, state


Deep = lambda cell: partial(DeepRNN, cell)

**Remark.** You can swap GRU out with any recurrent cell (e.g. RNN, LSTM).

In [ ]:
gru = Deep(GRU)(5, 2, num_layers=3)
print(gru)

Shape test:

In [ ]:
x = torch.randn(10, 32, 5)
outs, state = gru(x)
assert len(state) == 3
assert outs.shape == (10, 32, 2)
assert state[0].shape == (32, 2)

Correctness for layers more than 1:

In [ ]:
gru = Deep(GRU)(5, 2, num_layers=3)
gru_torch = nn.GRU(5, 2, num_layers=3)

for net in [gru, gru_torch]:
    for name, p in net.named_parameters():
        if "bias" in name:
            p.data.fill_(0.0)
        else:
            p.data.fill_(1.0)

error = torch.max(torch.abs(gru(x)[0] - gru_torch(x)[0]))
print(error)
assert error < 1e-5

Recovers base case (i.e. `num_layers=1`):

In [ ]:
gru = Deep(GRU)(5, 2, num_layers=1)
gru_base = GRU(5, 2)

for net in [gru, gru_base]:
    for name, p in net.named_parameters():
        if "bias" in name:
            p.data.fill_(0.0)
        else:
            p.data.fill_(1.0)

error = torch.max(torch.abs(gru_base(x)[0] - gru(x)[0]))
print(error)
assert error < 1e-6

<br>

### Model training

Common RNN layer widths are in the range [64, 2056] while depth is in [1, 8].

In [ ]:
VOCAB_SIZE = tokenizer.vocab_size
model = LanguageModel(Deep(GRU))(VOCAB_SIZE, 64, vocab_size=VOCAB_SIZE, num_layers=3)
model

In [ ]:
from tqdm.notebook import tqdm

DEVICE = "cpu"
LR = 0.01
EPOCHS = 5
MAX_NORM = 1.0

model.to(DEVICE)
optim = torch.optim.Adam(model.parameters(), lr=LR)

train_losses = []
valid_losses = []
for e in tqdm(range(EPOCHS)):
    for t, (x, y) in tqdm(enumerate(train_loader), total=len(train_loader)):
        x, y = x.to(DEVICE), y.to(DEVICE)
        loss = train_step(model, optim, x, y, MAX_NORM)
        train_losses.append(loss)

        if t % 5 == 0:
            xv, yv = next(iter(valid_loader))
            xv, yv = xv.to(DEVICE), yv.to(DEVICE)
            valid_losses.append(valid_step(model, xv, yv))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats("svg")

plt.figure(figsize=(10, 4))
plt.plot(train_losses, label="train")
plt.plot(np.array(range(1, len(valid_losses) + 1)) * 5, valid_losses, label="valid")
plt.grid(linestyle="dotted", alpha=0.6)
plt.ylabel("loss")
plt.xlabel("step")
plt.legend();

Best performance this chapter:

In [ ]:
np.array(valid_losses[-50:]).mean()

<br>

### Text generation

In [ ]:
textgen = TextGenerator(model, tokenizer, device="cpu")
s = [textgen.predict("thank y", num_preds=2, temperature=0.4) for i in range(20)]
(np.array(s) == "thank you").mean()

In [ ]:
warmup = "mr williams i underst"
text = []
temperature = []
for i in range(1, 6):
    t = 0.20 * i
    s = textgen.predict(warmup, num_preds=100, temperature=t)
    text.append(s)
    temperature.append(t)

In [ ]:
import pandas as pd
from IPython.display import display
pd.set_option("display.max_colwidth", None)
df = pd.DataFrame({"temp": [f"{t:.1f}" for t in temperature], "text": text})
df = df.style.set_properties(**{"text-align": "left"})
display(df)

This is surprisingly good. ꉂ૮(°□°'˶)ა

# Bidirectional Recurrent Units

For language modeling, i.e. predicting the next token, it makes sense to condition on the leftward context. However, it also makes sense to look at words from the right to fully understand a sentence. For example, a common task[^pretraining] is to mask out random tokens in a text document and then to train a sequence model to predict the values of the missing tokens. Depending on what comes after the blank, the likely value of the missing token changes dramatically:

```text
I am ___.
I am ___ tired.
I am ___ tired, and I can sleep all day.
```

Clearly the meaning change by reading more into the sentence.
The likely candidates change (e.g. "happy", "not", and "very" ), even when the tokens on the left 
of the blank are the same. Note that bidirectional RNNs typically only used in the *encoding* part of a system[^encoding] since the rightward context is not available at inference.

[^pretraining]: Often useful for model pretraining prior to fine-tuning on an actual task of interest.

[^encoding]: Encoding refers to ingesting an input sequence $\mathbf{x}$ to get a state vector $\mathbf{h}$ that, in principle, distills the relevant information in that sequence relative to the downstream task. For example, recall in text generation $\mathbf{h} = \text{RNN}(\mathbf{x}_{\text{prompt}})$ is the encoding used to start the *decoding* process, i.e. using the state vector and the model's generative capabilities to produce text.

<br>

## Bidirectional computation

A simple technique transforms any RNN unit (e.g. LSTM, GRU) into a bidirectional unit [@birnn]. 
The input sequence is simply processed in opposing directions. Then, outputs for the same inputs are concatenated for downstream processing. The states are processed independently in the expected order. 

More precisely, let $G$ be a recurrent unit. For $t = 1, \ldots, T$, the forward direction computes:


$$
\mathbf{O}^f_t, \mathbf{H}^f_t = G(\mathbf{X}_t, \mathbf{H}^f_{t-1})
$$

Meanwhile, the backward direction calculates: 

$$
\mathbf{O}^b_{T - t + 1}, \mathbf{H}^b_t = G(\mathbf{X}_{T - t + 1}, \mathbf{H}^b_{t-1})
$$

Finally, the output sequence is given by 

$$(\mathbf{O}^f_t \oplus \mathbf{O}^b_t)_{t=1}^T.$$ 

Note that input indices have to be matched. This is illustrated in @fig-05-birnn.

<br>

![Building the output vectors (right) in a bidirectional recurrent computation.](img/05-birnn.png){#fig-05-birnn fig-align="center" width=670px}

<br>

## Code implementation

In [ ]:
torch.manual_seed(42)
import certifi
import ssl
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

Creating a **Bidirectional** wrapper unit to convert any to a bidirectional RNN cell:

In [ ]:
from functools import partial

class BiRNN(RNNBase):
    def __init__(self, 
        cell: Type[RNNBase],
        inputs_dim: int, hidden_dim: int, 
        **kwargs
    ):
        super().__init__(inputs_dim, hidden_dim)
        assert hidden_dim % 2 == 0
        self.frnn = cell(inputs_dim, hidden_dim // 2, **kwargs)
        self.brnn = cell(inputs_dim, hidden_dim // 2, **kwargs)
        
    def init_state(self, x):
        return (None, None)

    def compute(self, x, state):
        fh, bh = state
        fo, fh = self.frnn(x, fh)
        bo, bh = self.brnn(torch.flip(x, [0]), bh)  # flip seq index: (T, B, d)
        bo = torch.flip(bo, [0])                    # flip back outputs. See above
        outs = torch.cat([fo, bo], dim=-1)
        return outs, (fh, bh)


Bidirectional = lambda cell: partial(BiRNN, cell)

**Remark.** For consistency with `LanguageModel`, we expect that the `hidden_dim` parameter is the size of the combined forward and backward outputs. This is so that the linear layer matches the given parameter. In other words, we halve the hidden size of each RNN cell inside the bidirectional model.

Flip function works as follows:

In [ ]:
l = [[i, 2 * i] for i in range(6)]
a = torch.tensor(l).reshape(-1, 3, 2)
a

Only time dimension is affected by flip:

In [ ]:
torch.flip(a, [0])

**Shapes.** Recall LSTM has two states. And one state vector for each input sequence:

In [ ]:
blstm = Bidirectional(LSTM)(28, 128)
x = torch.randn(30, 32, 28)
outs, (fstate, bstate) = blstm(x)
assert outs.shape == (30, 32, 128)
assert fstate[0].shape == (32, 64), fstate[1].shape == (32, 64)
assert bstate[0].shape == (32, 64), bstate[1].shape == (32, 64)

Our implementation differs a bit with PyTorch where hidden size is doubled:

In [ ]:
blstm_torch = nn.LSTM(28, 64, bidirectional=True)
for net in [blstm, blstm_torch]:
    for name, p in net.named_parameters():
        if "bias" in name:
            p.data.fill_(0.0)
        else:
            p.data.fill_(1.0)

error = torch.abs(blstm(x)[0] - blstm_torch(x)[0]).max()
print(error)
assert error < 5e-6

Adding depth still works:

In [ ]:
blstm = Deep(Bidirectional(LSTM))(28, 128, num_layers=2)
blstm_torch = nn.LSTM(28, 64, bidirectional=True, num_layers=2)
for net in [blstm, blstm_torch]:
    for name, p in net.named_parameters():
        if "bias" in name:
            p.data.fill_(0.0)
        else:
            p.data.fill_(1.0)

error = torch.abs(blstm(x)[0] - blstm_torch(x)[0]).max()
print(error)
assert error < 1e-5

**Remark.** *Bidirectional first, then deep.* We will train such a model below. This allows each layer to benefit from the bidirectional context before passing it to the next layer, i.e. the higher-order states also look at the previous states in two directions. The zero error above shows that this setup is consistent with the PyTorch implementation.

<br>

## Model training

**Demo.** Training a BiLSTM based sentiment classifier on the [IMDB movie review](https://keras.io/api/datasets/imdb/) dataset:

In [ ]:
import keras
import torch
from torch.utils.data import TensorDataset, DataLoader

def collate_fn(batch):
    """Transforming the data to sequence-first format."""
    x, y = zip(*batch)
    x = torch.stack(x, 1)      # (T, B)
    y = torch.stack(y, 0)      # (B,)
    return x, y


BATCH_SIZE = 16
MAX_WORDS = 20_000          # Only top MAX_WORDS words, else <unk>
T = 256                     # Only read T words of each movie review
PADDING_IDX = MAX_WORDS     # In practice, 0 may be <unk>, so new idx for padding

# Load. Trim or pad sequences to length T
(x_train, y_train), (x_valid, y_valid) = keras.datasets.imdb.load_data(num_words=MAX_WORDS)
x_train = keras.utils.pad_sequences(x_train, maxlen=T, value=PADDING_IDX)
x_valid = keras.utils.pad_sequences(x_valid, maxlen=T, value=PADDING_IDX)
print(x_train.shape, x_valid.shape)

train_dataset = TensorDataset(torch.tensor(x_train), torch.tensor(y_train).long())
valid_dataset = TensorDataset(torch.tensor(x_valid), torch.tensor(y_valid).long())
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)    # also sampled

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats("svg")

x = np.log(x_train[0] + 1)
plt.imshow(((x - x.min()) / (x.max() - x.min())).reshape(8, 32));

Observe that the sequence is **pre-padded**, i.e. padding token is added from the left[^padding]. This is good so that padding elements are processed before the RNN internal states are updated with information from actual inputs. 
Of course, it would be better if the RNN explicitly ignores the padding element (for efficiency).
Later it will be shown that LSTM networks and GRU are able to effectively ignore the padding vector during training.

[^padding]: RNNs can handle variable-length sequence inputs when $B = 1.$ However, to efficiently process batches of input, we have to have batches with the same shape $(T, B, d)$ hence the same sequence length. One way to do this is to **pack sequences**, where variable sized batches are constructed based on sequences that still have elements, so that padding is effectively ignored by the network. See this [SO answer](https://stackoverflow.com/a/56211056/1091950).

In [ ]:
from tqdm.notebook import tqdm

def train_step(model, optim, x, y, max_norm) -> float:
    loss = F.cross_entropy(model(x), y)
    loss.backward()
    
    model[0].weight.grad[PADDING_IDX] = 0.0
    clip_grad_norm(model, max_norm=max_norm)
    optim.step()
    optim.zero_grad()

    return loss.item()
 
@torch.no_grad()
def valid_step(model, x, y) -> float:
    loss = F.cross_entropy(model(x), y)
    return loss.item()

class LastElement(nn.Module):
    """Get last element of a rank-3 tensor of shape (T, B, h)."""
    def forward(self, x):
        return x[0][-1]


DEVICE = "mps"
LR = 1e-4
EPOCHS = 3
MAX_NORM = 5.0
EMBED_DIM = 512
HIDDEN_DIM = 128

emb = nn.Embedding(MAX_WORDS + 1, EMBED_DIM, padding_idx=PADDING_IDX)
rnn = Deep(Bidirectional(nn.LSTM))(EMBED_DIM, HIDDEN_DIM, num_layers=3, bias=False)
model = nn.Sequential(
    emb, rnn, LastElement(),
    nn.Dropout(0.2),
    nn.Linear(HIDDEN_DIM, 2)
)
model = model.to(DEVICE)
optim = torch.optim.Adam(model.parameters(), lr=LR)

train_loss = []
valid_loss = []
valid_accs = []

for e in tqdm(range(EPOCHS)):
    for t, (x, y) in tqdm(enumerate(train_loader), total=len(train_loader)):
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        loss = train_step(model, optim, x, y, MAX_NORM)
        train_loss.append(loss)

        if t % 5 == 0:
            xv, yv = next(iter(valid_loader))
            xv = xv.to(DEVICE)
            yv = yv.to(DEVICE)
            valid_loss.append(valid_step(model, xv, yv))
            valid_accs.append((model(xv).argmax(dim=1) == yv).float().mean().item())

**Remark.** Setting `padding_idx` in `nn.Embedding` makes the embedding vector for the padding token zero. Moreover, we set `bias=False` in the RNN. These constraints stabilized network training since the LSTM unit (also GRU) will have zero output for a sequence of padding tokens of any length[^lstm-no-bias]. This is consistent with adding the padding tokens to the left, so that network output is unaffected by the amount of left padding. Finally, gradients are leaking to the padding token due to BPTT, so we use the ff. heuristic to keep the padding embedding fixed:
```python
model[0].weight.grad[PADDING_IDX] = 0.0
```

[^lstm-no-bias]: This is an exact property of LSTM without bias: 
$\tilde{\mathbf{C}}_t=\tanh \left(\mathbf{X}_t^{\text{emb}} \mathbf{U}^c}+\mathbf{H}_{t-1} \mathbf{W}^c}\right) = \boldsymbol{0}$
whenever
$\mathbf{X}_t^{\text{emb}} = \boldsymbol{0}$ and $\mathbf{H}_{t-1} = \boldsymbol{0}.$ 
Moreover, ${\mathbf{H}}_t$ is also zero since it's the gated $\tanh\left( \mathbf{C}_t \right)$ and $\tanh 0 = 0.$ And because $\mathbf{H}_0 = \boldsymbol{0}$ and $\mathbf{C}_0 = \boldsymbol{0}$, the states do not update every time we get the padding token as input. By a similar calculation, the GRU can be shown to have this property.

Evaluating validation accuracy:

In [ ]:
c = 0
for xv, yv in tqdm(valid_loader):
    xv = xv.to(DEVICE)
    yv = yv.to(DEVICE)
    c += (model(xv).argmax(dim=1) == yv).int().sum().item()

print(c / len(valid_dataset))

In [ ]:
fig, ax1 = plt.subplots(figsize=(6, 4))

WINDOW_SIZE = int(0.05 * len(valid_loss))
train_loss = np.array(train_loss)
valid_loss = np.array(valid_loss)
valid_accs = np.array(valid_accs)
train_loss_avg = [train_loss[i-WINDOW_SIZE * 5:i].mean() for i in range(WINDOW_SIZE * 5, len(train_loss))]
valid_loss_avg = [valid_loss[i-WINDOW_SIZE:i].mean() for i in range(WINDOW_SIZE, len(valid_loss))]
valid_accs_avg = [valid_accs[i-WINDOW_SIZE:i].mean() for i in range(WINDOW_SIZE, len(valid_accs))]

ax1.plot(train_loss_avg, linewidth=2, alpha=0.6, label="train loss", color="C0")
ax1.set_xlabel("step")
ax1.grid(axis="both", linestyle="dotted", alpha=0.8)
ax2 = ax1.twinx()
ax1.plot(np.array(range(1, len(valid_loss_avg) + 1)) * 5, valid_loss_avg, linewidth=2, label="valid loss", color="C0")
ax2.plot(np.array(range(1, len(valid_accs_avg) + 1)) * 5, valid_accs_avg, linewidth=2, label="valid accs", color="C1")

ax1.set_ylabel("loss", fontsize=10)
ax2.set_ylabel("accuracy", fontsize=10)
ax1.xaxis.set_label_coords(1.00, -0.020)
ax1.legend(loc="lower right", fontsize=10)
ax2.legend(loc="upper right", fontsize=10)
ax2.legend(loc='upper center', bbox_to_anchor=(0.1, -0.10), ncol=5)
ax1.legend(loc='lower center', bbox_to_anchor=(0.8, -0.20), ncol=5);

Checking trained model outputs zero vector on padding tokens:

In [ ]:
z = (torch.ones(256, 1).to(DEVICE) * PADDING_IDX).long()
out, state = model[1](model[0](z))

# forward is zero
error = (out.abs() - torch.zeros(256, 1, 128).to(DEVICE)).mean().item()
assert error == 0.0

# internal states are zero
for l in range(model[1].num_layers):
    (hf, cf), (hb, cb) = state[l]
    assert hf.sum() + cf.sum() == 0
    assert hb.sum() + cb.sum() == 0

# Appendix: Numerical stability

From the BP equations, the key quantity that affects the numerical stability is $\frac{\partial\mathcal{L}}{\partial\mathbf{H}_t}$ @eq-state_vec_grad. Suppose $\mathbf{W}$ has a diagonalization[^diagonalizable] $\mathbf{W} = \mathbf{Q}\boldsymbol{\Lambda} \mathbf{Q}^{-1}$ where $\boldsymbol{\Lambda} = \text{diag}(\lambda_1, \ldots, \lambda_h)$ with $|\lambda_1| > \ldots > |\lambda_h|$, then 

$$
\mathbf{W}^\kappa = \mathbf{Q}\boldsymbol{\Lambda}^\kappa \mathbf{Q}^{-1}.
$$

Hence, the *principal eigenvalue* $\lambda_1 \in \mathbb{C}$ dominates:

$$
\mathbf{W}^\kappa = \lambda_1^\kappa\;
    \mathbf{Q}
    \left[
        \begin{array}{llll}
            1 & & & \\
            & \left(\frac{\lambda_2}{\lambda_1}\right)^\kappa & & \\
            & & \ddots & \\
            & & & \left(\frac{\lambda_h}{\lambda_1}\right)^\kappa
        \end{array}
    \right] 
    \mathbf{Q}^{-1} \to \; \lambda_1^\kappa\; \mathbf{Q}\left[\begin{array}{llll}
1 & & & \\
& 0 & & \\
& & \ddots & \\
& & & 0
\end{array}\right] \mathbf{Q}^{-1}
$$

as $\kappa \to \infty.$ 
The rightmost terms terms in the matrix product are fixed. If $|\lambda_1| > 1,$ the product diverges, while it vanishes to zero when $|\lambda_1| < 1.$ Finally, $|\lambda_1| = 1$ occurs with zero probability. The first two cases are verified in code below:

[^diagonalizable]: A random matrix with entries in $U[a, b]$ is [diagonalizable with probability 1](https://www.imsc.res.in/~kapil/papers/matrix/index.html)  over $\mathbb{C}$.

In [ ]:
import numpy as np
np.random.seed(10)

eps = 1e-5
norms = {
    1.0 - eps: [],
    1.0 + eps: []
}

for c in norms.keys():

    # Rescale principal eigenvalue of A ~ N(0, 1)
    W = np.random.rand(10, 10)
    λ = np.linalg.norm(np.linalg.eig(W).eigenvalues[0])
    W = W / λ * c

    N = 18
    for i in range(N):
        norms[c].append(np.linalg.norm(W))
        W = W @ W

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats("svg")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
K = range(3, N)
for j, c in enumerate(norms.keys()):
    ax[j].plot(K, [norms[c][k] for k in K], color=f"C{j}", linewidth=2)
    ax[j].set_ylabel(r"$\|\; \boldsymbol{\mathbf{W}}^k \;\|$")
    ax[j].set_xlabel(r"$\kappa$")
    ax[j].set_title(r"$|\lambda_1| =$" + f" {c:.5f}")

fig.tight_layout();